# 🚀 Training XLM-RoBERTa on UIT-VSFC (Cleaned Text + Class Weights + Threshold Tuning)

Notebook này nâng cấp pipeline huấn luyện **XLM-RoBERTa** trên tập dữ liệu **UIT-VSFC** với 3 giải pháp cải tiến:
1. **Pre-processing (Tiền xử lý)**: Áp dụng `clean_text_vietnamese()` xử lý rác `doubledot`, `11doubledot55` -> `11:55`, `wzjwz<id>` -> `[ANON]`, `fraction` -> `/` cho cả 3 tập **Train, Validation, Test**.
2. **Weighted Cross-Entropy Loss**: Gán trọng số nghịch đảo tần suất lớp để khắc phục class imbalance nặng cho nhãn **NEUTRAL** (~4% dataset).
3. **Post-processing Threshold Tuning**: Tối ưu ngưỡng xác suất cho lớp NEUTRAL (hạ $\tau \approx 0.13$) để tăng mạnh Recall & F1-score.


In [3]:
# Cell 1: Environment Setup & Clone Repository
import os
os.chdir('/kaggle/working')  # Về thư mục làm việc gốc Kaggle

# Clone repo hoặc pull bản mới nhất
!git clone -b model/sentiment-training-setup https://github.com/nhienthai/AI_in_DevOps-DataOps-MLOps_Final_Project.git 2>/dev/null || (cd AI_in_DevOps-DataOps-MLOps_Final_Project && git pull)

os.chdir('/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project')
print('📍 Current Working Directory:', os.getcwd())

# Fix pip setup.py issue trên Kaggle
!pip install --upgrade pip setuptools wheel -q
!pip install --no-cache-dir datasets transformers accelerate mlflow -q


Already up to date.
📍 Current Working Directory: /kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 39.1 MB/s eta 0:00:00


In [4]:
# Cell 2: Preprocess Dataset với clean_text_vietnamese cho cả 3 tập Train/Val/Test
import re
import pandas as pd
from datasets import load_dataset, DatasetDict

print('🚀 Loading dataset tridm/UIT-VSFC...')
ds = load_dataset('tridm/UIT-VSFC')

def clean_text_vietnamese(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # 1. Thay 'doubledot' -> ':' (xử lý cả 11doubledot55 -> 11:55)
    text = re.sub(r'doubledot', ':', text, flags=re.IGNORECASE)
    # 2. Thay 'fraction' -> '/'
    text = re.sub(r'\bfraction\b', '/', text, flags=re.IGNORECASE)
    # 3. Thay 'wzjwz<id>' -> '[ANON]'
    text = re.sub(r'wzjwz\d+', '[ANON]', text, flags=re.IGNORECASE)
    return text.strip()

print('🧹 Applying clean_text_vietnamese to train, validation, and test splits...')
cleaned_ds = DatasetDict({
    split: ds[split].map(lambda ex: {'Sentence': clean_text_vietnamese(ex['Sentence'])})
    for split in ds.keys()
})

print('--- Sample comparison (Raw vs Cleaned) ---')
print('Raw:    ', ds['train'][0]['Sentence'])
print('Cleaned:', cleaned_ds['train'][0]['Sentence'])


🚀 Loading dataset tridm/UIT-VSFC...


train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/11426 [00:00<?, ? examples/s]

...

🧹 Applying clean_text_vietnamese to train, validation, and test splits...


Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

Map:   0%|          | 0/3166 [00:00<?, ? examples/s]

--- Sample comparison (Raw vs Cleaned) ---
Raw:     slide giáo trình đầy đủ .
Cleaned: slide giáo trình đầy đủ .


In [5]:
# Cell 3: Fine-Tune XLM-RoBERTa với Weighted CrossEntropy Loss & Cosine Schedule
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model_name = 'xlm-roberta-base'
max_length = 128
batch_size = 16
epochs = 10
learning_rate = 2e-5
output_dir = './artifacts/xlm-roberta'

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['Sentence'], padding='max_length', truncation=True, max_length=max_length)

tokenized_ds = cleaned_ds.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.rename_column('Encoded_sentiment', 'label')

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Tính Class Weights khắc phục lệch nhãn NEUTRAL (~4%)
label_counts = Counter(cleaned_ds['train']['Encoded_sentiment'])
total = sum(label_counts.values())
num_classes = 3
class_weights = torch.tensor(
    [total / (num_classes * label_counts.get(i, 1)) for i in range(num_classes)],
    dtype=torch.float
)
print('⚖️ Computed Class Weights:', class_weights.tolist())

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        device = next(model.parameters()).device
        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Starting Fine-Tuning...')
trainer.train()

# Lưu model & tokenizer
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print('✅ Model saved to', output_dir)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./artifacts/xlm-roberta


In [6]:
# Cell 4: Đánh giá & So sánh Argmax vs. Threshold Tuning trên tập Test
from sklearn.metrics import classification_report

print('📊 Evaluating model on Test set...')
test_preds_raw = trainer.predict(tokenized_ds['test'])
test_logits = test_preds_raw.predictions
test_labels = test_preds_raw.label_ids
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()

# 1. Dự đoán bằng Standard Argmax (Ngưỡng 0.33)
preds_argmax = np.argmax(test_probs, axis=1)
print('=== 1. PREDICT WITH ARGMAX (Standard Softmax) ===')
print(classification_report(test_labels, preds_argmax, target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'], digits=4))

# 2. Dự đoán bằng Threshold Tuning cho NEUTRAL (tau_NEUTRAL = 0.13)
def predict_with_threshold(probs, tau_neutral=0.13):
    preds = []
    for p in probs:
        p_neg, p_neu, p_pos = p[0], p[1], p[2]
        if p_neu >= tau_neutral:
            preds.append(1) # NEUTRAL
        else:
            preds.append(0 if p_neg >= p_pos else 2)
    return np.array(preds)

preds_threshold = predict_with_threshold(test_probs, tau_neutral=0.13)
print('=== 2. PREDICT WITH THRESHOLD TUNING (tau_NEUTRAL = 0.13) ===')
print(classification_report(test_labels, preds_threshold, target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'], digits=4))


📊 Evaluating model on Test set...


=== 1. PREDICT WITH ARGMAX (Standard Softmax) ===
              precision    recall  f1-score   support

    NEGATIVE     0.9384    0.9730    0.9554      1409
     NEUTRAL     0.7000    0.4611    0.5560       167
    POSITIVE     0.9524    0.9553    0.9538      1590

    accuracy                         0.9371      3166
   macro avg     0.8636    0.7965    0.8217      3166
weighted avg     0.9328    0.9371    0.9336      3166

=== 2. PREDICT WITH THRESHOLD TUNING (tau_NEUTRAL = 0.13) ===
              precision    recall  f1-score   support

    NEGATIVE     0.9403    0.9723    0.9560      1409
     NEUTRAL     0.6752    0.4731    0.5563       167
    POSITIVE     0.9529    0.9541    0.9535      1590

    accuracy                         0.9368      3166
   macro avg     0.8561    0.7998    0.8220      3166
weighted avg     0.9326    0.9368    0.9337      3166



In [7]:
# Cell 5: Nén Model Weights & Xuất Link Tải File trên Kaggle
os.chdir('/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project')
!zip -r model_weights.zip ./artifacts/xlm-roberta
!mv model_weights.zip /kaggle/working/ 2>/dev/null || true

os.chdir('/kaggle/working')
from IPython.display import FileLink, display
print('📦 Model weights zipped successfully!')
display(FileLink('model_weights.zip'))


  adding: artifacts/xlm-roberta/ (stored 0%)
  adding: artifacts/xlm-roberta/tokenizer_config.json (deflated 47%)
  adding: artifacts/xlm-roberta/config.json (deflated 53%)
  adding: artifacts/xlm-roberta/checkpoint-3580/ (stored 0%)
  adding: artifacts/xlm-roberta/checkpoint-3580/trainer_state.json (deflated 74%)
  adding: artifacts/xlm-roberta/checkpoint-3580/rng_state.pth (deflated 26%)
  adding: artifacts/xlm-roberta/checkpoint-3580/config.json (deflated 53%)
  adding: artifacts/xlm-roberta/checkpoint-3580/scaler.pt (deflated 64%)
  adding: artifacts/xlm-roberta/checkpoint-3580/optimizer.pt (deflated 71%)
  adding: artifacts/xlm-roberta/checkpoint-3580/model.safetensors (deflated 26%)
  adding: artifacts/xlm-roberta/checkpoint-3580/training_args.bin (deflated 53%)
  adding: artifacts/xlm-roberta/checkpoint-3580/scheduler.pt (deflated 61%)
  adding: artifacts/xlm-roberta/checkpoint-3222/ (stored 0%)
  adding: artifacts/xlm-roberta/checkpoint-3222/trainer_state.json (deflated 74%)
  

/kaggle/working/model_weights.zip